# imports

In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import yaml
import os
import sys
import datetime
import time

import argparse
from pathlib import Path
from copy import deepcopy

from torchtoolbox.optimizer import CosineWarmupLr

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from utils.general import select_device, increment_path, Logger, AverageMeter, print_argument_options, init_torch_seeds, is_parallel

from models.build import build_models, HeadAndLoss
# from evaluation import evals
from models.iresnet import IResNet
import torch.serialization
import json


# Data Loading

In [2]:
class CustomDataset(datasets.ImageFolder):
    def __init__(self, root, transform=None, save_dir=None):
        super(CustomDataset, self).__init__(root, transform=transform)
        if save_dir:
            with open(save_dir / 'class_to_idx.json', 'w') as f:
                json.dump(self.class_to_idx, f)

def build_datasets(data_path, batch_size, cuda, workers, mode='train', rank=-1, save_dir=None):
    data_dir = data_path
    transform = transforms.Compose([
        transforms.Resize((112, 112)),  # Resize images to the input size expected by the model
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize images
    ])

    dataset = CustomDataset(root=data_dir, transform=transform, save_dir=save_dir if mode == 'train' else None)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(mode == 'train'),
        num_workers=workers,
        pin_memory=cuda
    )

    return dataset, loader

# Training

In [3]:
def train(opt, model, headandloss, optimizer, scheduler, trainloader, device, rank):
    model.train()
    losses = AverageMeter()        
    meta_dict = {'snmin': AverageMeter(), 'snmax': AverageMeter(), 
                 'spwmin': AverageMeter(), 'spwmax': AverageMeter(),
                 'snwmin': AverageMeter(), 'snwmax': AverageMeter()}

    start_time = time.time() 
    for i, (data, labels) in enumerate(trainloader):                        
        data, labels = data.to(device), labels.to(device)          
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            deep_features = model(data)                  
            loss, metas = headandloss(deep_features, labels, rank)

        # Ensure loss is a scalar
        if loss.dim() > 0:  # If loss is not a scalar, reduce it
            loss = loss.mean()

        # Backpropagation
        if device.type == 'cuda':
            opt['scaler'].scale(loss).backward()  # Use scaler from opt dictionary
            opt['scaler'].step(optimizer)
            opt['scaler'].update()
        else:
            loss.backward()
            optimizer.step()

        scheduler.step()  # Update learning rate
        
        # Update metadata
        for key, val in metas.items():            
            meta_dict[key].update(val, labels.size(0)) 
        losses.update(loss.item(), labels.size(0))        

        # Print training progress
        if (i + 1) % opt['print_freq'] == 0 and rank in [-1, 0]:
            elapsed = str(datetime.timedelta(seconds=round(time.time() - start_time)))
            start_time = time.time()            
            s = ''
            for key, val in meta_dict.items():
                s += key + ' {:.6f} ({:.6f}) '.format(val.val, val.avg)
            print("Batch {}/{}\t Loss {:.6f} ({:.6f}), {} elapsed time (h:m:s): {}" \
                .format(i + 1, len(trainloader), losses.val, losses.avg, s, elapsed))

# Main

In [4]:
def main(opt):
    # Device setup
    if torch.backends.mps.is_available():  # Check if Metal (MPS) is available
        device = torch.device('mps')  # Use Metal backend on Apple Silicon
    elif torch.cuda.is_available():  # Check if CUDA is available
        device = torch.device('cuda')  # Use CUDA backend on NVIDIA GPUs
    else:
        device = torch.device('cpu')  # Fallback to CPU

    print(f"####################################### -------------------->  Using device: {device}")

    # Create save directory
    save_dir = Path(opt['save_dir'])
    save_dir.mkdir(parents=True, exist_ok=True)

    # Logging
    sys.stdout = Logger(save_dir / 'log_.txt', opt['resume'])

    # Initialize random seeds
    init_torch_seeds(opt['seed'] + 1 + opt['global_rank'])

    # Build training dataset and data loader
    train_dataset, train_loader = build_datasets(opt['data']['data_dir'], opt['batch_size'], device.type != 'cpu', opt['workers'], mode='train', rank=opt['global_rank'], save_dir=save_dir)


    # Build model
    model = build_models(opt['model'], drop_lastfc=opt['drop_lastfc']).to(device)
    criterion = nn.CrossEntropyLoss()
    headandloss = HeadAndLoss(512, opt['data']['num_classes'], criterion, opt['head'], opt['aux'], head_zoo=opt['head_zoo']).to(device)
    

    # Save the head settings
    with open(save_dir / 'head_cfg.yaml', 'w') as f:
        yaml.dump(headandloss.head_cfg, f, sort_keys=False)
    if opt['aux']:
        with open(save_dir / 'aux_cfg.yaml', 'w') as f:
            yaml.dump(headandloss.aux_cfg, f, sort_keys=False)

    # Load pre-trained weights (if any)
    pretrained = opt['weights'].endswith('.pt')
    if pretrained:
        ckpt = torch.load(opt['weights'], map_location=device)
        model_state_dict = ckpt['backbone'].float().state_dict()
        model.load_state_dict(model_state_dict, strict=False)
        if ckpt.get('headandloss') is not None:
            head_state_dict = ckpt['headandloss'].float().state_dict()
            headandloss.load_state_dict(head_state_dict, strict=False)
            del head_state_dict

    # Distributed training setup (if applicable)
    if device.type == 'cuda' and opt['global_rank'] == -1 and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
        headandloss.head = torch.nn.DataParallel(headandloss.head)
        if opt['aux']:
            headandloss.aux = torch.nn.DataParallel(headandloss.aux)

    if device.type == 'cuda' and opt['global_rank'] != -1:
        model = DDP(model, device_ids=[opt['local_rank']], output_device=opt['local_rank'])
        headandloss = DDP(headandloss, device_ids=[opt['local_rank']], output_device=opt['local_rank'])

    # Print model and head information
    if opt['global_rank'] in [-1, 0]:
        print("Create model  : {}".format(opt['model']))
        print("Create head   : {}".format(opt['head']))

    # Optimizer and scheduler setup
    optimizer = torch.optim.SGD(model.parameters(), lr=opt['hyp']['lr'], weight_decay=opt['hyp']['weight_decay'], momentum=opt['hyp']['momentum'])
    optimizer.add_param_group({'params': headandloss.parameters()})

    batches_per_epoch = len(train_loader)
    scheduler = CosineWarmupLr(optimizer, batches_per_epoch, opt['max_epoch'], base_lr=opt['hyp']['lr'], warmup_epochs=opt['hyp']['warmup_epochs'])

    # Gradient scaler for mixed precision training
    if device.type == 'cuda':
        scaler = torch.cuda.amp.GradScaler(enabled=True)  # Use CUDA GradScaler
    elif device.type == 'mps':
        scaler = None  # Metal (MPS) does not support GradScaler
    else:
        scaler = None  # CPU does not support GradScaler
        
    # Add scaler to the opt dictionary
    opt['scaler'] = scaler

    # Resume training (if applicable)
    best_acc = 0.0
    if pretrained and opt['resume']:
        if ckpt.get('optimizer') is not None:
            optimizer.load_state_dict(ckpt['optimizer'])
        if ckpt.get('scheduler') is not None:
            scheduler.load_state_dict(ckpt['scheduler'])
        best_acc = ckpt['best_acc']
        opt['start_epoch'] = ckpt['epoch'] + 1
        del ckpt, model_state_dict

    # Training loop
    for epoch in range(opt['start_epoch'], opt['max_epoch']):
        if opt['global_rank'] != -1:
            train_loader.sampler.set_epoch(epoch)
        if opt['global_rank'] in [-1, 0]:
            print("==> Epoch {}/{}".format(epoch + 1, opt['max_epoch']))

        # Train for one epoch
        train(opt, model, headandloss, optimizer, scheduler, train_loader, device, opt['global_rank'])

        # Save checkpoint
        if opt['global_rank'] in [-1, 0]:
            ckpt = {
                'epoch': epoch,
                'best_acc': best_acc,
                'backbone': deepcopy(model.module if is_parallel(model) else model).eval(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict()
            }
            if len(headandloss.state_dict().keys()) > 0:
                ckpt['headandloss'] = deepcopy(headandloss.module if is_parallel(headandloss) else headandloss).eval()
            else:
                ckpt['headandloss'] = None
            # Ensure the 'weights' directory exists
            weights_dir = save_dir / 'weights'
            weights_dir.mkdir(parents=True, exist_ok=True)

            # Now save the checkpoint
            torch.save(ckpt, weights_dir / 'last.pt')
            
            if best_acc == ckpt['best_acc']:
                torch.save(ckpt, save_dir / 'weights' / 'best.pt')
            del ckpt

In [5]:
# Hardcoded options (replace with your desired values)
opt = {
        'save_dir': 'runs/exp3',  # Directory to save logs and checkpoints
        'data': {
            'data_dir': 'main_data/train/',  # Path to training dataset
            'img_size': 224,  # Image size ----------------> we can try 224 
            'num_classes': 126  # Number of classes (persons)
        },
        'hyp': {
            'lr': 0.001,  # Learning rate
            'weight_decay': 0.0005,  # Weight decay
            'momentum': 0.7,  # Momentum
            'warmup_epochs': 3.0 , # Warmup epochs
            "gamma": 0.1
        },
        'model': 'iresnet-100',  # Model architecture
        'head': 'arcface',  # Head type
        'aux': '',  # Auxiliary loss (if any)
        'drop_lastfc': False,  # Whether to drop the last fully connected layer
        'weights': 'face.r100.cos.unpg.wisk1.5.pt',  # Path to pre-trained weights (if any)
        'resume': False,  # Whether to resume training
        'batch_size': 64,  # Batch size
        'max_epoch': 40,  # Maximum number of epochs
        'start_epoch': 0,  # Starting epoch
        'workers': 0,  # Number of data loading workers
        'seed': 1000,  # Random seed
        'global_rank': -1,  # Global rank for distributed training
        'local_rank': -1,  # Local rank for distributed training
        'local_test': False,  # Whether to perform local testing
        'head_zoo': 'head.zoo.yaml',  # Path to head configuration file
        'print_freq':40
        
    }

In [ ]:
# Start training
main(opt)

####################################### -------------------->  Using device: cuda
{'arcface': {'s': 64.0, 'm': 0.5}}
Head config file arcface
{'arcface': {'s': 64.0, 'm': 0.5}}

Aux config file 
{'': None}
Create model  : iresnet-100
Create head   : arcface
==> Epoch 1/40


C:\Users\3sol\AppData\Local\Temp\ipykernel_12960\2116447446.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(opt['weights'], map_location=device)
C:\Us

Batch 40/107	 Loss 38.787109 (38.794977), snmin -0.185669 (-0.207483) snmax 0.355469 (0.361841) spwmin -0.098572 (-0.116144) spwmax 0.314209 (0.397049) snwmin -0.185669 (-0.207483) snwmax 0.355469 (0.361841)  elapsed time (h:m:s): 0:00:18
Batch 80/107	 Loss 36.313721 (38.130743), snmin -0.237305 (-0.213071) snmax 0.447998 (0.369751) spwmin -0.064880 (-0.115184) spwmax 0.391846 (0.402939) snwmin -0.237305 (-0.213071) snwmax 0.447998 (0.369751)  elapsed time (h:m:s): 0:00:16
==> Epoch 2/40
Batch 40/107	 Loss 26.473097 (29.454128), snmin -0.267822 (-0.256284) snmax 0.343750 (0.418542) spwmin 0.013885 (-0.097451) spwmax 0.701172 (0.590857) snwmin -0.267822 (-0.256284) snwmax 0.343750 (0.418542)  elapsed time (h:m:s): 0:00:15
Batch 80/107	 Loss 14.117559 (24.918832), snmin -0.263184 (-0.258244) snmax 0.360596 (0.419742) spwmin 0.002163 (-0.051034) spwmax 0.807617 (0.655823) snwmin -0.263184 (-0.258244) snwmax 0.360596 (0.419742)  elapsed time (h:m:s): 0:00:15
==> Epoch 3/40
Batch 40/107	 Lo

# inference 

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os
import json

# Load the trained model
def load_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = checkpoint['backbone']
    model.eval()  # Set the model to evaluation mode
    return model

# Load class index mapping from JSON file
def load_class_mapping(json_path):
    with open(json_path, 'r') as f:
        class_to_idx = json.load(f)
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    return idx_to_class

# Preprocess the input image
def preprocess_image(image_path, transform):
    image = Image.open(image_path).convert('RGB')
    image = transform(image)
    image = image.unsqueeze(0)  # Add batch dimension
    return image

# Perform inference
def infer(model, image, device, idx_to_class):
    with torch.no_grad():
        image = image.to(device)
        output = model(image)
        _, predicted = torch.max(output, 1)
        predicted_class = predicted.item()

        # Check if the predicted class exists in the training dataset
        return idx_to_class.get(predicted_class, "doesn't exist")

# Main function to test the model on real data
def test_model(model, test_dir, transform, device, idx_to_class):
    results = {}
    for image_name in os.listdir(test_dir):
        image_path = os.path.join(test_dir, image_name)
        image = preprocess_image(image_path, transform)
        predicted_label = infer(model, image, device, idx_to_class)
        results[image_name] = predicted_label
    return results

# Define the transformation
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # Resize images to the input size expected by the model
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Normalize images
])

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"CUDA is available: {cuda_available}")
# Define the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the trained model
checkpoint_path = 'runs/exp2/weights/last.pt'  # Path to the best checkpoint
model = load_model(checkpoint_path, device)
print("after loading model")

# Load class mappings
class_mapping_path = 'runs/exp2/class_to_idx.json'  # Path to the saved class mapping
idx_to_class = load_class_mapping(class_mapping_path)
print("Loaded idx_to_class mappings")

# Test the model on real data
test_dir = 'main_data/test/'   # Path to the test directory
results = test_model(model, test_dir, transform, device, idx_to_class)
print("test done")

# Print the results
for image_name, predicted_label in results.items():
    print(f"Image: {image_name}, Predicted Label: {predicted_label}")


C:\Users\3sol\AppData\Local\Temp\ipykernel_18236\2366613935.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)

In [ ]:
image_path = "main_data/train/person_0/9.jpg"
image = preprocess_image(image_path, transform)
predicted_label = infer(model, image, device, idx_to_class)
predicted_label

'person_15'